In [1]:
import pandas as pd
import numpy as np
import os
import glob

# ==========================================================
# НАСТРОЙКИ: Укажите пути здесь (или оставьте пустыми, 
# тогда скрипт попросит ввести их при запуске)
# ==========================================================
SVOD_PATH = "/home/jovyan/old_home/Litigant/ahanter_fio_parsing/СВОД.xlsx"   # Например: r"C:\docs\СВОД.xlsx"
FOLDER_PATH = "/home/jovyan/old_home/Litigant/ahanter_fio_parsing/Filial_tables" # Например: r"C:\docs\other_files"
# ==========================================================

In [2]:
def clean_excel_string(series):
    """Очищает столбцы от артефактов Excel"""
    s = series.astype(str).str.replace(r'\.0$', '', regex=True)
    s = s.replace('nan', None)
    return s

def normalize_columns(df):
    """Приводит названия столбцов к единому виду"""
    rename_map = {}
    for col in df.columns:
        col_str = str(col).strip()
        col_lower = col_str.lower()
        if col_lower == 'фио': rename_map[col] = 'ФИО'
        elif col_lower in ['дата рождения', 'датарождения', 'дата_рождения']: rename_map[col] = 'ДАТА РОЖДЕНИЯ'
        elif col_lower == 'инн': rename_map[col] = 'ИНН'
        elif col_lower in ['телефон', 'phone', 'tel', 'номер телефона']: rename_map[col] = 'ТЕЛЕФОН'
        elif col_lower in ['e-mail', 'email', 'e_mail', 'почта', 'mail', 'электронная почта']: rename_map[col] = 'E-MAIL'
        elif col_lower in ['адрес', 'address', 'адрес проживания']: rename_map[col] = 'АДРЕС'
            
    df.rename(columns=rename_map, inplace=True)
    if 'ИНН' in df.columns: df['ИНН'] = clean_excel_string(df['ИНН'])
    if 'ТЕЛЕФОН' in df.columns: df['ТЕЛЕФОН'] = clean_excel_string(df['ТЕЛЕФОН'])
    return df

def format_birth_date(df, col_name):
    """Приводит дату рождения к формату дд.мм.гггг"""
    if col_name not in df.columns: return df
    def to_ddmmyyyy(val):
        if pd.isna(val): return val
        try:
            dt = pd.to_datetime(val, dayfirst=True, errors='coerce')
            if pd.isna(dt): return str(val)
            return dt.strftime('%d.%m.%Y')
        except Exception: return str(val)
    df[col_name] = df[col_name].apply(to_ddmmyyyy)
    return df

def process_files(svod_path, folder_path):
    target_cols = ["ФИО", "ДАТА РОЖДЕНИЯ", "ИНН", "ТЕЛЕФОН", "E-MAIL", "АДРЕС"]

    print(f"Загрузка основного файла: {svod_path}")
    svod_df = pd.read_excel(svod_path)
    svod_df = normalize_columns(svod_df)
    svod_df = format_birth_date(svod_df, 'ДАТА РОЖДЕНИЯ')

    original_columns = svod_df.columns.tolist()

    # Определяем, какие ключи доступны для сопоставления
    has_inn = 'ИНН' in svod_df.columns
    has_fio_dr = 'ФИО' in svod_df.columns and 'ДАТА РОЖДЕНИЯ' in svod_df.columns
    
    print(f"Доступные ключи для сопоставления: {'ИНН' if has_inn else ''} {'ФИО+ДР' if has_fio_dr else ''}".strip())

    # Ищем все Excel файлы в папке
    excel_files = glob.glob(os.path.join(folder_path, '*.xlsx')) + glob.glob(os.path.join(folder_path, '*.xls'))
    svod_abs_path = os.path.abspath(svod_path)
    output_abs_path = os.path.abspath(os.path.join(os.path.dirname(svod_path), 'СВОД.xlsx'))
    excel_files = [f for f in excel_files if os.path.abspath(f) not in (svod_abs_path, output_abs_path)]

    if not excel_files:
        print("В папке не найдено других Excel-файлов.")
        return

    print(f"Найдено файлов для обработки: {len(excel_files)}")

    # Счетчик для сохранения порядка строк
    svod_df['_row_order'] = range(len(svod_df))

    for file in excel_files:
        print(f"-> Обработка {os.path.basename(file)}...")
        try:
            other_df = pd.read_excel(file)
            other_df = normalize_columns(other_df)
            other_df = format_birth_date(other_df, 'ДАТА РОЖДЕНИЯ')

            # Выравниваем столбцы: добавляем недостающие из СВОД
            for col in svod_df.columns:
                if col not in other_df.columns:
                    other_df[col] = np.nan
                    other_df = other_df[[c for c in svod_df.columns if c in other_df.columns or c in target_cols]]

            # === ШАГ 1: Дозаполнение по ИНН ===
            if has_inn and 'ИНН' in other_df.columns:
                other_with_inn = other_df[other_df['ИНН'].notna()].copy()
                if not other_with_inn.empty:
                    other_with_inn.set_index('ИНН', inplace=True)
                    other_with_inn = other_with_inn[~other_with_inn.index.duplicated(keep='first')]
                    
                    svod_with_inn = svod_df[svod_df['ИНН'].notna()].copy()
                    svod_with_inn.set_index('ИНН', inplace=True)
                    
                    svod_with_inn = svod_with_inn.combine_first(other_with_inn)
                    svod_with_inn.reset_index(inplace=True)
                    
                    # Обновляем свод
                    svod_df = svod_df[svod_df['ИНН'].isna() | ~svod_df['ИНН'].isin(svod_with_inn['ИНН'])]
                    svod_df = pd.concat([svod_df, svod_with_inn], ignore_index=True)

            # === ШАГ 2: Дозаполнение по ФИО+ДАТА РОЖДЕНИЯ ===
            if has_fio_dr and 'ФИО' in other_df.columns and 'ДАТА РОЖДЕНИЯ' in other_df.columns:
                other_df['_key_fio_dr'] = other_df['ФИО'].astype(str) + '_' + other_df['ДАТА РОЖДЕНИЯ'].astype(str)
                svod_df['_key_fio_dr'] = svod_df['ФИО'].astype(str) + '_' + svod_df['ДАТА РОЖДЕНИЯ'].astype(str)
                
                other_with_fio_dr = other_df[other_df['_key_fio_dr'].notna()].copy()
                if not other_with_fio_dr.empty:
                    other_with_fio_dr.set_index('_key_fio_dr', inplace=True)
                    other_with_fio_dr = other_with_fio_dr[~other_with_fio_dr.index.duplicated(keep='first')]
                    
                    svod_with_fio_dr = svod_df[svod_df['_key_fio_dr'].notna()].copy()
                    svod_with_fio_dr.set_index('_key_fio_dr', inplace=True)
                    
                    svod_with_fio_dr = svod_with_fio_dr.combine_first(other_with_fio_dr)
                    svod_with_fio_dr.reset_index(inplace=True)
                    
                    svod_df = svod_df[svod_df['_key_fio_dr'].isna() | ~svod_df['_key_fio_dr'].isin(svod_with_fio_dr['_key_fio_dr'])]
                    svod_df = pd.concat([svod_df, svod_with_fio_dr], ignore_index=True)
                    
                    svod_df.drop(columns=['_key_fio_dr'], inplace=True)

            # === ШАГ 3: Добавление новых строк ===
            # Находим строки в other_df, которые не совпали ни по ИНН, ни по ФИО+ДР
            new_rows = other_df.copy()
            
            if has_inn and 'ИНН' in new_rows.columns:
                new_rows = new_rows[new_rows['ИНН'].isna() | ~new_rows['ИНН'].isin(svod_df['ИНН'].dropna())]
            
            if has_fio_dr and 'ФИО' in new_rows.columns and 'ДАТА РОЖДЕНИЯ' in new_rows.columns:
                new_rows['_key_fio_dr'] = new_rows['ФИО'].astype(str) + '_' + new_rows['ДАТА РОЖДЕНИЯ'].astype(str)
                svod_df['_key_fio_dr'] = svod_df['ФИО'].astype(str) + '_' + svod_df['ДАТА РОЖДЕНИЯ'].astype(str)
                new_rows = new_rows[new_rows['_key_fio_dr'].isna() | ~new_rows['_key_fio_dr'].isin(svod_df['_key_fio_dr'].dropna())]
                new_rows.drop(columns=['_key_fio_dr'], inplace=True)
                svod_df.drop(columns=['_key_fio_dr'], inplace=True)
            
            # Оставляем только строки, где есть хоть какие-то данные в целевых столбцах
            new_rows = new_rows[new_rows[target_cols].notna().any(axis=1)]
            
            if not new_rows.empty:
                max_order = svod_df['_row_order'].max()
                start_order = int(max_order) + 1 if not pd.isna(max_order) else 0
                new_rows['_row_order'] = range(start_order, start_order + len(new_rows))
                svod_df = pd.concat([svod_df, new_rows], ignore_index=True)
                print(f"   Добавлено новых строк: {len(new_rows)}")

        except Exception as e:
            print(f"   Ошибка при обработке файла {file}: {e}")

    # Восстанавливаем порядок строк
    svod_df.sort_values('_row_order', inplace=True)
    svod_df.drop(columns=['_row_order'], inplace=True)

    # Восстанавливаем порядок столбцов
    final_columns = [col for col in original_columns if col in svod_df.columns]
    svod_df = svod_df[final_columns]

    output_dir = os.path.dirname(svod_path)
    output_path = os.path.join(output_dir, 'СВОД.xlsx')
    
    print(f"\nСохранение обновленного файла: {output_path}")
    svod_df.to_excel(output_path, index=False)
    print("Готово!")

In [3]:
svod_path = SVOD_PATH.strip()
folder_path = FOLDER_PATH.strip()
if not svod_path:
    svod_path = input("Введите полный путь к файлу СВОД.xlsx: ").strip().strip('"')
if not folder_path:
    folder_path = input("Введите полный путь к папке с другими Excel-файлами: ").strip().strip('"')
if not os.path.exists(svod_path):
    print(f"Ошибка: Файл '{svod_path}' не найден.")
    input("Нажмите Enter для выхода...")
    exit(1)
if not os.path.isdir(folder_path):
    print(f"Ошибка: Папка '{folder_path}' не найдена.")
    input("Нажмите Enter для выхода...")
    exit(1)
process_files(svod_path, folder_path)

Загрузка основного файла: /home/jovyan/old_home/Litigant/ahanter_fio_parsing/СВОД.xlsx
Доступные ключи для сопоставления: ИНН ФИО+ДР
Найдено файлов для обработки: 4
-> Обработка Свод_сутяжники_0.xlsx...
-> Обработка Копия Свод_сутяжники (004).xlsx...
-> Обработка Свод_сутяжники_1.xlsx...
-> Обработка Свод_сутяжники.xlsx...

Сохранение обновленного файла: /home/jovyan/old_home/Litigant/ahanter_fio_parsing/СВОД.xlsx
Готово!
